In [1]:
import operator
import random
from typing import Annotated, Sequence, TypedDict, List, Dict

from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, ToolMessage

from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

from langgraph.managed.is_last_step import RemainingSteps
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError
from langgraph_supervisor.supervisor import create_supervisor


/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_9212/611322112.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.sql_database import SQLDatabase


In [2]:
LOCAL_LLM = 'gemma4:12b-mlx'
TEMPERATURE = 0.7
llm_model = ChatOllama(
    model=LOCAL_LLM,
    temperature=TEMPERATURE,
    use_responses_api=True
)

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps

# Travel Agent.

In [3]:
# Travel info store.
SAVE_DIR = './chroma_travel_db'
EMBEDDING_MODEL = 'nomic-embed-text:latest'

vectorstore_client = Chroma(
    persist_directory=SAVE_DIR,
    embedding_function=OllamaEmbeddings(model=EMBEDDING_MODEL)
)
retriever = vectorstore_client.as_retriever()

WEATHER: Sequence[str] = ['sunny', 'foggy', 'rainy', 'windy']
class WeatherForecast(TypedDict):
    town: str
    weather: Literal = WEATHER
    temperature: int
    
@tool(description='Get the weather forecast given the town name.')
def weather_forecast(town: str) -> dict:
    '''Get a weather forecast for a given town.
    Returns a WeatherForecast object with weather and temperature.
    '''
    _weather_options = WEATHER
    _temp_min = 18
    _temp_max = 31
    
    weather = random.choice(_weather_options)
    temperature = random.randint(_temp_min, _temp_max)
    return WeatherForecast(town=town, weather=weather, temperature=temperature)
    
@tool(description='Search travel information about destinations in England.')
def search_travel_info(query: str) -> str:
    """Search embedded WikiVoyage content for 
    information about destinations in England."""    
    docs = retriever.invoke(query)
    top = docs[:4] if isinstance(docs, list) else docs
    return "\n---\n".join(d.page_content for d in top)

agent_prompt = '''
You are a helpful travel assistant that searches information and retrieves weather forecasts.    

CRITICAL RULES:
    1. Only suggest destinations that you have found inside the 'search_travel_info' tool.
    2. Identify candidate towns from your travel info search and check the weather for MULTIPLE candidate towns in parallel (simultaneously) to find the ones with the best weather.
    3. If your initial batch of towns has bad weather, query the weather for any backup towns in a single batch before formulating your final answer.
'''
travel_agent = create_react_agent(
    name='Travel Agent',
    model=llm_model,
    tools=[weather_forecast, search_travel_info],
    state_schema=AgentState,
    prompt=agent_prompt,
)


/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_9212/2391970278.py:46: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  travel_agent = create_react_agent(


# Booking Agent.

In [4]:
hotel_db = SQLDatabase.from_uri('sqlite:///cornwall_hotels.db')
hotel_db_toolkit = SQLDatabaseToolkit(db=hotel_db, llm=llm_model)
hotel_db_toolkit_tools = hotel_db_toolkit.get_tools()

class BnBOffer(TypedDict):
    # Define the return type of the BnB availability tool
    bnb_id: int
    bnb_name: str
    town: str
    available_rooms: int
    price_per_room: float

class BnBBookingService:
    # Define the BnB availability tool
    @staticmethod
    # Call the BnB booking service to get the offers
    def get_offers_near_town(town: str, num_rooms: int) -> List[BnBOffer]:
        # Mocked REST API response: multiple BnBs per destination
        mock_bnb_offers = [ # Mocked BnB offers
            # Newquay
            {"bnb_id": 1, "bnb_name": "Seaside BnB", 
            "town": "Newquay", "available_rooms": 3, 
            "price_per_room": 80.0},
            {"bnb_id": 2, "bnb_name": "Surfside Guesthouse", 
            "town": "Newquay", "available_rooms": 2, 
            "price_per_room": 85.0},
            # Falmouth
            {"bnb_id": 3, "bnb_name": "Harbour View BnB", 
            "town": "Falmouth", "available_rooms": 4, 
            "price_per_room": 78.0},
            {"bnb_id": 4, "bnb_name": "Seafarer's Rest", 
            "town": "Falmouth", "available_rooms": 1, 
            "price_per_room": 90.0},
            # St Austell
            {"bnb_id": 5, "bnb_name": "Garden Gate BnB", 
            "town": "St Austell", "available_rooms": 2, "price_per_room": 82.0},
            {"bnb_id": 6, "bnb_name": "Coastal Cottage BnB", 
            "town": "St Austell", "available_rooms": 3, "price_per_room": 88.0},
            # Penzance
            {"bnb_id": 7, "bnb_name": "Penzance Pier BnB", 
            "town": "Penzance", "available_rooms": 2, "price_per_room": 95.0},
            {"bnb_id": 8, "bnb_name": "Cornish Charm BnB", 
            "town": "Penzance", "available_rooms": 3, "price_per_room": 87.0},
            # Camborne
            {"bnb_id": 9, "bnb_name": "Camborne Corner BnB", 
            "town": "Camborne", "available_rooms": 2, "price_per_room": 75.0},
            {"bnb_id": 10, "bnb_name": "Rose Cottage BnB", 
            "town": "Camborne", "available_rooms": 2, "price_per_room": 79.0},
            # Hayle
            {"bnb_id": 11, "bnb_name": "Hayle Haven BnB", 
            "town": "Hayle", "available_rooms": 3, "price_per_room": 83.0},
            {"bnb_id": 12, "bnb_name": "Dune View BnB", 
            "town": "Hayle", "available_rooms": 1, "price_per_room": 81.0},
            # Land's End
            {"bnb_id": 13, "bnb_name": "Land's End Lookout BnB", 
            "town": "Land's End", "available_rooms": 2, "price_per_room": 100.0},
            {"bnb_id": 14, "bnb_name": "Atlantic Edge BnB", 
            "town": "Land's End", "available_rooms": 2, "price_per_room": 105.0},
            # Bude
            {"bnb_id": 15, "bnb_name": "Bude Beach BnB", 
            "town": "Bude", "available_rooms": 2, "price_per_room": 77.0},
            {"bnb_id": 16, "bnb_name": "Cliffside BnB", 
            "town": "Bude", "available_rooms": 3, "price_per_room": 80.0},
            # Padstow
            {"bnb_id": 17, "bnb_name": "Padstow Harbour BnB", 
            "town": "Padstow", "available_rooms": 2, "price_per_room": 92.0},
            {"bnb_id": 18, "bnb_name": "Fisherman's Rest BnB", 
            "town": "Padstow", "available_rooms": 2, "price_per_room": 89.0},
            # St Ives
            {"bnb_id": 19, "bnb_name": "St Ives Bay BnB", "town": "St Ives", "available_rooms": 3, "price_per_room": 97.0},
            {"bnb_id": 20, "bnb_name": "Artists' Retreat BnB", "town": "St Ives", "available_rooms": 2, "price_per_room": 102.0},
            # Looe
            {"bnb_id": 21, "bnb_name": "Looe Riverside BnB", "town": "Looe", "available_rooms": 2, "price_per_room": 84.0},
            {"bnb_id": 22, "bnb_name": "Harbour Lights BnB", "town": "Looe", "available_rooms": 2, "price_per_room": 86.0},
            # Polperro
            {"bnb_id": 23, "bnb_name": "Polperro Cove BnB", "town": "Polperro", "available_rooms": 2, "price_per_room": 91.0},
            {"bnb_id": 24, "bnb_name": "Smuggler's Rest BnB", "town": "Polperro", "available_rooms": 2, "price_per_room": 93.0},
            # Mevagissey
            {"bnb_id": 25, "bnb_name": "Mevagissey Harbour BnB", "town": "Mevagissey", "available_rooms": 2, "price_per_room": 90.0},
            {"bnb_id": 26, "bnb_name": "Seafarer's BnB", "town": "Mevagissey", "available_rooms": 2, "price_per_room": 88.0},
            # Port Isaac
            {"bnb_id": 27, "bnb_name": "Port Isaac View BnB", 
            "town": "Port Isaac", "available_rooms": 2, 
            "price_per_room": 99.0},
            {"bnb_id": 28, "bnb_name": "Fisherman's Cottage BnB", 
            "town": "Port Isaac", "available_rooms": 2, 
            "price_per_room": 101.0},
            # Fowey
            {"bnb_id": 29, "bnb_name": "Fowey Quay BnB", 
            "town": "Fowey", "available_rooms": 2, 
            "price_per_room": 94.0},
            {"bnb_id": 30, "bnb_name": "Riverside Rest BnB", 
            "town": "Fowey", "available_rooms": 2, 
            "price_per_room": 96.0},
        ]
        offers = [
            offer for offer in mock_bnb_offers 
            if offer["town"].lower() == town.lower()
            and offer["available_rooms"] >= num_rooms
        ]
        return offers

@tool(description='''Check BnB room availability and price for a destination in Cornwall.''')
def check_bnb_availability(destination_town: str, num_rooms: int) -> List[Dict]:
    """Check BnB room availability and price for a specific town.
    
    Args:
        destination_town: The specific town name (e.g., 'Newquay', 'Falmouth', 'St Ives'). 
        num_rooms: The number of required rooms.
    """
    return BnBBookingService.get_offers_near_town(destination_town, num_rooms)    

booking_prompt = '''
You are a dual-engine holiday accommodation matcher.

CRITICAL FORCING RULES:
1. To answer completely, you MUST search for BOTH hotels (using sql_db_query) AND bed & breakfasts (using 'check_bnb_availability').
2. Do NOT stop or reply after running a SQL query. You must immediately call 'check_bnb_availability' for the same town before writing your final response.
3. Output a single combined Markdown table containing BOTH types of accommodations.
'''

booking_agent = create_react_agent(
    name='Booking Agent',
    model=llm_model,
    tools=hotel_db_toolkit_tools + [check_bnb_availability],
    state_schema=AgentState,
    prompt=booking_prompt # Direct string avoids the array-append reduction bug
)


/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_9212/2112155360.py:122: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  booking_agent = create_react_agent(


# Orchestrator.

In [5]:
system_prompt = '''
You are a strict supervisor managing two specialized agents:
1. 'Travel Agent' (Call this agent first to find towns in Cornwall and check weather).
2. 'Booking Agent' (Call this agent second using the towns found by the Travel Agent to look up accommodations).

CRITICAL OPERATIONAL RULES:
1. When a user asks for BOTH a town and accommodation availability, you MUST call BOTH agents before providing a final answer to the user.
2. Once the Travel Agent returns candidate towns (even if the weather is rainy/foggy), immediately transfer control to the Booking Agent for those towns. 
3. Do not ask the user follow-up questions halfway through the workflow. Deliver a complete package: town weather AND accommodation availability.
4. Even if the weather is bad, present the towns you found so that the booking agent can check for accommodations anyway. Do not halt the graph to ask for backup towns unless explicitly told 'only show sunny options'.

'''
travel_assistant = create_supervisor(
    agents=[travel_agent, booking_agent],
    model= llm_model,
    supervisor_name="travel_assistant",
    prompt=system_prompt,
    ).compile()

In [6]:
def chat_loop():
    print("Booking Assistant (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        state = {"messages": [HumanMessage(content=user_input)]}
        
        config = {"recursion_limit": 30} 
        
        try:
            result = travel_assistant.invoke(state, config=config)
            print("\n\n\n")
            print(result)
            print("\n\n\n")
            response_msg = result["messages"][-1].content
            print(f"Assistant: {response_msg}\n")
        except GraphRecursionError:
            # Catch the limit gracefully if it hits a runaway loop
            print("\nAssistant: I'm sorry, I couldn't find any accomodation.\n")

            

In [7]:
chat_loop()

Booking Assistant (type 'exit' to quit)


You:  Can you find a nice seaside Cornwall town with good weather and find availability for one hotel room?






{'messages': [HumanMessage(content='Can you find a nice seaside Cornwall town with good weather and find availability for one hotel room?', additional_kwargs={}, response_metadata={}, id='fe31e750-591c-4a83-82b3-e5365b43bc8a'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-mlx', 'created_at': '2026-06-12T05:38:22.525627Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6892022417, 'load_duration': 1438550500, 'prompt_eval_count': 299, 'prompt_eval_duration': 518349292, 'eval_count': 147, 'eval_duration': 4933652416, 'logprobs': None, 'model_name': 'gemma4:12b-mlx', 'model_provider': 'ollama'}, name='travel_assistant', id='lc_run--019eba56-8fcf-7012-94d1-6481ff321ed8-0', tool_calls=[{'name': 'transfer_to_travel_agent', 'args': {}, 'id': '9fe7c48c-8d3f-4fd1-82f4-ff13473718c2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 299, 'output_tokens': 147, 'total_tokens': 446}), ToolMessage(content='Successfully tr

You:  exit
